# Токенайзер

In [1]:
from nltk.tokenize import TweetTokenizer

In [2]:
from string import punctuation

In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
from nltk.corpus import stopwords

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
def tokenizer(t):
  t = t.lower()
  tokeniz = TweetTokenizer()
  tokeny = tokeniz.tokenize(t)
  tokeny = [tok for tok in tokeny if (tok not in stopwords.words('russian')) and (tok not in punctuation)]
  return tokeny

In [7]:
from nltk.stem.snowball import SnowballStemmer

In [8]:
st = SnowballStemmer('russian')

In [48]:
tok = tokenizer('Попытка сделать хакатон по ТП')

In [49]:
final = [st.stem(x) for x in tok]
final

['попытк', 'сдела', 'хакатон', 'тп']

# Чтение файлов из папки для выявления ключевых слов, прогнанных через токенайзер


In [14]:
def search_for_sender(file):
  for line in file:
    if 'From' in line or 'from' in line or 'Ot kogo' in line or 'От кого' in line:
      words = line.split()
      for mail in words:
        if '@' in mail:
          if '<' in mail and '>' in mail:
            return mail[1:-1]
          else:
            return mail

Извлечение из письма отправителя письма

In [13]:
def search_for_recipient(file):
  for line in file:
    if 'to' in line or 'Komu' in line or 'Кому' in line or 'To' in line:
      words = line.split()
      for mail in words:
        if '@' in mail:
          return mail
  else:
    return None

Извлечение из письма получателя письма

In [54]:
with open('/content/HAKATON/inbox/mail_0001.txt', 'r', encoding='utf-8') as file:
    content1 = file.readlines()
    print(content1)

['Subject: браузер Chrome зависает при открытии\n', 'From: s.volkov@partner.ru\n', '\n', 'Здравствуйте!\n', '\n', 'После обновления системы браузер Chrome не открывает файлы нужного формата. Раньше всё работало.\n', '\n', 'Спасибо.']


In [66]:
def search_subject(file):
  k = 0
  for line in file:
    if 'Subject' in line or 'subject' in line:
      words = line.split()
      k+=1
    if k == 1:
      return line

In [67]:
search_subject(content1)

'Subject: браузер Chrome зависает при открытии\n'

In [68]:
def search_text(file):
  text = ''
  for line in file:
    sender = search_for_sender(file)
    recipient = search_for_recipient(file)
    subject = search_subject(file)
    sender_valid = sender not in line
    recipient_valid = recipient is None or recipient not in line
    subject_valid = search_subject not in line
    if sender_valid and recipient_valid and subject_valid:
      text += f'{line}'
  return text

Извлечение текста из письма, нужного для определения категории, в которую будет определено письмо


In [50]:
categories_weights = {
    'Спам': {
        'выигра': 0.08, 'приз': 0.08, 'iphone': 0.08, 'exclusive': 0.08, 'offer': 0.08,
        'limited': 0.06, 'time': 0.04, 'личност': 0.05, 'верификац': 0.03, 'аккаунт': 0.03,
        'заблокирова': 0.06, 'перейд': 0.06, 'ссылк': 0.06, 'password-reset': 0.05,
        'secure-login': 0.05, 'сброс': 0.03, 'парол': 0.03, 'стран': 0.03, 'активн': 0.03
    },
    'Важное': {
        'urgent': 0.12, 'срочн': 0.12, 'критическ': 0.12, 'массов': 0.09, 'сбо': 0.12,
        'работ': 0.08, 'остановл': 0.09, 'отвеча': 0.08, 'недоступ': 0.08, 'утечк': 0.05,
        'дан': 0.02, 'краж': 0.03
    },
    'Технические ошибки': {
        'ошибк': 0.07, 'error': 0.07, 'err': 0.07, 'зависа': 0.06, 'слома': 0.06,
        'ddos': 0.05, 'ремонт': 0.05, 'перебо': 0.05, 'интернет': 0.04, 'связ': 0.04,
        'запуска': 0.04, 'обновлен': 0.04, 'открыва': 0.04, 'компьютер': 0.03, 'ноутбук': 0.03,
        'принтер': 0.03, 'сканер': 0.03, 'outlook': 0.03, 'chrome': 0.03, 'excel': 0.03,
        'zoom': 0.03, 'электричеств': 0.03, 'помех': 0.02, 'мыш': 0.02, 'гарнитур': 0.02
    },
    'Информация': {
        'дайджест': 0.11, 'отчет': 0.09, 'мониторинг': 0.11, 'info': 0.09, 'healthcheck': 0.11,
        'созвон': 0.07, 'дем': 0.11, 'обновлен': 0.05, 'планов': 0.07, 'работ': 0.03,
        'брифинг': 0.07, 'брейншторм': 0.06, 'встреч': 0.03
    },
    'Документы': {
        'договор': 0.17, 'акт': 0.17, 'закрыва': 0.17, 'документ': 0.14, 'согласован': 0.12,
        'contract': 0.12, 'финальн': 0.07, 'верс': 0.02, 'дан': 0.02
    },
    'Счета': {
        'invoice': 0.12, 'реквизит': 0.11, 'оплат': 0.10, 'счет': 0.10, 'задолжен': 0.09,
        'выписк': 0.08, 'кред': 0.07, 'выплат': 0.06, 'бухгалтер': 0.06, 'страховк': 0.06,
        'зарплат': 0.05, 'компенсац': 0.04, 'прем': 0.06
    },
    'Подтверждение доступа к аккаунту': {
        'vpn': 0.11, 'gitlab': 0.11, 'confluence': 0.11, '1c': 0.11, 'выда': 0.09,
        'прав': 0.09, 'восстанов': 0.08, 'запрос': 0.07, 'доступ': 0.07, 'логин': 0.05,
        'аккаунт': 0.04, 'почт': 0.04, 'парол': 0.03
    },
    'HR': {
        'отпуск': 0.10, 'больничн': 0.10, 'резюм': 0.09, 'оформлен': 0.08, 'должност': 0.08,
        'назначен': 0.08, 'повышен': 0.08, 'отпускн': 0.08, 'опоздан': 0.07, 'нетрудоспособн': 0.07,
        'график': 0.05, 'сотрудник': 0.05, 'нов': 0.04, 'работ': 0.02, 'период': 0.01
    }
}

Разбиение слов, прогнанных через токенайзер по весу слов. Сумма слов в категории дает 1.00, что дает равные шансы для попадания письма в каждую категорию, что дает более качественный результат.
